# Ablations A1-A3 on Math (Week 3)

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. Use **Save & Run All (Commit)**. Profiles all 3,639 Math docs + queries three times (main run was ~16 GPU-min; expect ~1 h wall in total including the vLLM install).

When done, download `ablation_out.zip` from the Output tab, unzip it in the project folder (`unzip -n`), then run `python scripts/run_ablations.py` locally.

Generated by `scripts/build_kaggle_notebook.py --ablation`; don't edit by hand.

In [ ]:
!pip install -q vllm sentence-transformers huggingface_hub
!pip uninstall -y -q torchaudio
import torch; print(torch.__version__, torch.cuda.device_count(), 'GPUs')

In [ ]:
import os
for d in sorted({os.path.dirname(f) for f in ['src/obliq/__init__.py', 'src/obliq/data.py', 'src/obliq/profiles.py', 'scripts/profile_corpus.py', 'prompts/math_problem.txt', 'prompts/math_problem_short.txt', 'prompts/math_problem_solution.txt', 'prompts/math_problem_topic.txt', 'prompts/rewrite_math.txt', 'prompts/rewrite_writing.txt', 'prompts/writing_style.txt']}):
    os.makedirs(f'/kaggle/working/irproj/{d}', exist_ok=True)
%cd /kaggle/working/irproj

In [ ]:
%%writefile src/obliq/__init__.py
# package marker


In [ ]:
%%writefile src/obliq/data.py
"""Load OBLIQ-Bench tasks (corpus, queries, gold/pooled qrels, per-query exclusions)."""
import csv
import json
from dataclasses import dataclass, field
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
DATA = ROOT / "data" / "obliq"
TASK_DIRS = {"math": "analogues/math", "writing": "analogues/writing"}

Qrels = dict[str, dict[str, int]]


@dataclass
class Task:
    name: str
    corpus: dict[str, str]
    queries: dict[str, str]
    qrels: Qrels                      # gold judgments
    qrels_pool: Qrels | None          # pooled judgments (Math only)
    excluded: dict[str, set[str]] = field(default_factory=dict)


def download(tasks=("math", "writing")):
    from huggingface_hub import snapshot_download
    snapshot_download("dianetc/OBLIQ-Bench", repo_type="dataset", local_dir=DATA,
                      allow_patterns=[f"{TASK_DIRS[t]}/*" for t in tasks] + ["README.md"])


def _read_jsonl(path: Path) -> dict[str, str]:
    with open(path, encoding="utf-8") as f:
        return {d["_id"]: d["text"] for d in map(json.loads, f) if d}


def _read_qrels(path: Path) -> Qrels | None:
    # Tab-separated with a header; column names differ per task (query-id vs query_id).
    if not path.exists():
        return None
    qrels: Qrels = {}
    with open(path, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)
        for qid, did, score in reader:
            if int(score) > 0:
                qrels.setdefault(qid, {})[did] = int(score)
    return qrels


def load_task(name: str) -> Task:
    base = DATA / TASK_DIRS[name]
    qdir = base / "queries+qrels"
    if not (base / "corpus" / "corpus.jsonl").exists():
        download((name,))
    excl_path = qdir / "per_query_excluded_ids.json"
    excluded = {}
    if excl_path.exists():
        excluded = {q: set(ids) for q, ids in json.loads(excl_path.read_text()).items()}
    return Task(
        name=name,
        corpus=_read_jsonl(base / "corpus" / "corpus.jsonl"),
        queries=_read_jsonl(qdir / "queries.jsonl"),
        qrels=_read_qrels(qdir / "qrels.tsv"),
        qrels_pool=_read_qrels(qdir / "qrels_pool.tsv"),
        excluded=excluded,
    )


In [ ]:
%%writefile src/obliq/profiles.py
"""Latent profiles: build profiling prompts, generate with an LLM, cache to disk.

Cache layout: profiles/<task>/<variant>/{docs,queries}.jsonl with {"_id", "profile"} per line,
plus meta.json recording the prompt hash, model and generation settings. A variant's cache is
never mixed with output from a different prompt or model.
"""
import hashlib
import json
import re
from pathlib import Path

from .data import DATA, ROOT, Task

PROMPTS = ROOT / "prompts"
PROFILES = ROOT / "profiles"

MATH_Q_PREFIX = "Given the following mathematical problem:\n\n"
MATH_Q_SUFFIX_START = "\n\nFind other mathematical problems that require the same core reasoning pattern"

# Which prompt each task uses for documents and queries (queries never see solutions).
TASK_PROMPTS = {
    "writing": {"doc": "writing_style", "query": "writing_style"},
    "math": {"doc": "math_problem", "query": "math_problem"},
    "math_soln": {"doc": "math_problem_solution", "query": "math_problem"},
    # Ablations (Math only): A1 topic words allowed, A3 ~40-word profiles. A2 (3B profiler) reuses
    # "math" with another model; the model name is part of the cache folder.
    "math_topic": {"doc": "math_problem_topic", "query": "math_problem_topic"},
    "math_short": {"doc": "math_problem_short", "query": "math_problem_short"},
    # E2 query rewriters (queries only; "doc" is unused but keeps the variant hash well-defined).
    # Writing is the paper's authorship rewriter prompt verbatim (App. D); the paper gives no Math
    # rewriter prompt, so Math follows its published Twitter rewriter template.
    "writing_rewrite": {"doc": "rewrite_writing", "query": "rewrite_writing"},
    "math_rewrite": {"doc": "rewrite_math", "query": "rewrite_math"},
}

MAX_CHARS = 4000  # truncate long Writing snippets; style is visible well before this


def strip_math_query(text: str) -> str:
    """Math queries wrap a problem in fixed boilerplate; profile only the problem."""
    assert text.startswith(MATH_Q_PREFIX), text[:80]
    return text[len(MATH_Q_PREFIX):].split(MATH_Q_SUFFIX_START)[0].strip()


def load_math_solutions() -> dict[str, dict[str, str]]:
    rows = json.loads((DATA / "analogues/math/final_dataset.json").read_text(encoding="utf-8"))
    return {f"{r['competition']}___{r['year']}___{r['problem_id']}": r for r in rows}


def read_prompt(name: str) -> str:
    return (PROMPTS / f"{name}.txt").read_text(encoding="utf-8")


def build_prompts(task: Task, kind: str, ids: list[str], setting: str | None = None) -> dict[str, str]:
    """User-turn prompt text for each id. kind is 'doc' or 'query'."""
    setting = setting or task.name
    template = read_prompt(TASK_PROMPTS[setting][kind])
    if kind == "query":
        src = task.queries
        clean = strip_math_query if task.name == "math" else (lambda t: t)
        return {i: template.format(text=clean(src[i])[:MAX_CHARS]) for i in ids}
    if setting == "math_soln":
        sol = load_math_solutions()
        return {i: template.format(problem=sol[i]["problem"], solution=sol[i]["solution"][:MAX_CHARS])
                for i in ids}
    return {i: template.format(text=task.corpus[i][:MAX_CHARS]) for i in ids}


# ---------------------------------------------------------------- generation backends

class HFBackend:
    """Local transformers backend (Windows-friendly; used for hand-testing on the laptop GPU)."""

    # transformers has no output-only frequency penalty; block long verbatim loops instead.
    decoding = "greedy-norepeat8"

    def __init__(self, model: str, max_new_tokens: int = 220, batch_size: int = 4):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(model, padding_side="left")
        self.model = AutoModelForCausalLM.from_pretrained(model, device_map="cuda:0", torch_dtype=torch.float16)
        self.max_new_tokens, self.batch_size = max_new_tokens, batch_size

    def generate(self, prompts: list[str]) -> list[str]:
        import torch
        out = []
        for b in range(0, len(prompts), self.batch_size):
            chats = [self.tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                                  add_generation_prompt=True)
                     for p in prompts[b:b + self.batch_size]]
            enc = self.tok(chats, return_tensors="pt", padding=True).to(self.model.device)
            with torch.no_grad():
                gen = self.model.generate(**enc, max_new_tokens=self.max_new_tokens, do_sample=False,
                                          no_repeat_ngram_size=8, pad_token_id=self.tok.pad_token_id)
            out += self.tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        return [o.strip() for o in out]


class VLLMBackend:
    """Batched vLLM backend (Kaggle / Linux GPU) for profiling whole corpora."""

    # Greedy decoding looped on some Writing profiles ("metabolic signatures" x30); a frequency
    # penalty on generated tokens only (not the prompt) stops that without blocking quotes.
    decoding = "greedy-freqpen0.3"

    def __init__(self, model: str, max_new_tokens: int = 220, **llm_kwargs):
        from vllm import LLM, SamplingParams
        # T4s have no bf16; one T4 (16 GB) holds the 7B AWQ model, leaving the 2nd GPU free.
        kw = {"dtype": "half", "max_model_len": 4096, "gpu_memory_utilization": 0.85, "seed": 0}
        self.llm = LLM(model=model, **{**kw, **llm_kwargs})
        self.params = SamplingParams(temperature=0.0, frequency_penalty=0.3, max_tokens=max_new_tokens)

    def generate(self, prompts: list[str]) -> list[str]:
        msgs = [[{"role": "user", "content": p}] for p in prompts]
        return [o.outputs[0].text.strip() for o in self.llm.chat(msgs, self.params, use_tqdm=True)]


# ---------------------------------------------------------------- cache

def _meta(task_setting: str, kind: str, model: str, max_new_tokens: int, decoding: str) -> dict:
    prompt = read_prompt(TASK_PROMPTS[task_setting][kind])
    return {"prompt": TASK_PROMPTS[task_setting][kind],
            "prompt_sha": hashlib.sha256(prompt.encode()).hexdigest()[:12],
            "model": model, "max_new_tokens": max_new_tokens, "max_chars": MAX_CHARS,
            "decoding": decoding}


def variant_name(setting: str, model: str, decoding: str) -> str:
    """Cache folder name; changes whenever a prompt, the model or the decoding changes."""
    p = TASK_PROMPTS[setting]
    key = read_prompt(p["doc"]) + read_prompt(p["query"]) + decoding
    return f"{setting}-{hashlib.sha256(key.encode()).hexdigest()[:6]}__{model.split('/')[-1]}"


CACHE_FILES = {"doc": "docs.jsonl", "query": "queries.jsonl"}


_ECHO = re.compile(r"\n\s*(Passage|Problem|Solution)\s*:.*", re.S)


def clean_profile(text: str) -> str:
    """The model sometimes appends the source text after its answer ("Passage: ..."); drop it.
    Applied on read, so the raw cache keeps exactly what the model generated."""
    return _ECHO.sub("", text).strip()


def _cache_paths(vdir: Path, kind: str) -> list[Path]:
    """Main cache file first, then any shard files left by parallel (one-per-GPU) runs."""
    stem = CACHE_FILES[kind].removesuffix(".jsonl")
    return [p for p in [vdir / CACHE_FILES[kind], *sorted(vdir.glob(f"{stem}.shard*.jsonl"))] if p.exists()]


def _read_rows(path: Path) -> dict[str, str]:
    rows = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                continue  # half-written last line from a killed run; that id is regenerated
            rows[d["_id"]] = d["profile"]
    return rows


def load_profiles(task: str, variant: str, kind: str, raw: bool = False) -> dict[str, str]:
    rows = {}
    for path in _cache_paths(PROFILES / task / variant, kind):
        rows.update(_read_rows(path))
    return rows if raw else {i: clean_profile(t) for i, t in rows.items()}


def _append(path: Path):
    """Open a cache file for appending, first ending any unterminated last line (e.g. a file
    written by Jupyter's %%writefile, or a run killed mid-write) so new rows start on their own line."""
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as f:
            f.seek(-1, 2)
            if f.read(1) != b"\n":
                with open(path, "a", encoding="utf-8") as g:
                    g.write("\n")
    return open(path, "a", encoding="utf-8")


def merge_shards(task: str, variant: str, kind: str) -> None:
    """Fold shard files into the main cache file (skipping ids already there), then delete them."""
    vdir = PROFILES / task / variant
    main = vdir / CACHE_FILES[kind]
    have = _read_rows(main) if main.exists() else {}
    shards = [p for p in _cache_paths(vdir, kind) if p != main]
    if not shards:
        return  # also covers a brand-new variant whose folder doesn't exist yet
    with _append(main) as f:
        for sp in shards:
            for i, text in _read_rows(sp).items():
                if i not in have:
                    have[i] = text
                    f.write(json.dumps({"_id": i, "profile": text}, ensure_ascii=False) + "\n")
    for sp in shards:
        sp.unlink()


def profile(task: Task, kind: str, ids: list[str], backend, variant: str, model: str,
            setting: str | None = None, max_new_tokens: int = 220, chunk: int = 256,
            shard: int | None = None) -> dict[str, str]:
    """Generate profiles for ids not yet cached; append to the cache as each chunk finishes.
    With shard=k, new rows go to <kind>.shard<k>.jsonl so parallel processes never share a file."""
    setting = setting or task.name
    vdir = PROFILES / task.name / variant
    vdir.mkdir(parents=True, exist_ok=True)
    meta_path = vdir / f"meta_{kind}.json"
    meta = _meta(setting, kind, model, max_new_tokens, backend.decoding)
    if meta_path.exists():
        old = json.loads(meta_path.read_text())
        assert old == meta, f"cache {vdir} was built with different settings:\n{old}\nvs\n{meta}"
    else:
        # atomic, since parallel shards may write it at the same moment
        tmp = meta_path.with_suffix(f".tmp{shard or 0}")
        tmp.write_text(json.dumps(meta, indent=1))
        tmp.replace(meta_path)

    done = load_profiles(task.name, variant, kind)
    todo = [i for i in ids if i not in done]
    prompts = build_prompts(task, kind, todo, setting)
    out = vdir / (CACHE_FILES[kind] if shard is None else CACHE_FILES[kind].replace(".jsonl", f".shard{shard}.jsonl"))
    with _append(out) as f:
        for c in range(0, len(todo), chunk):
            part = todo[c:c + chunk]
            for i, text in zip(part, backend.generate([prompts[i] for i in part])):
                done[i] = clean_profile(text)
                f.write(json.dumps({"_id": i, "profile": text}, ensure_ascii=False) + "\n")
            f.flush()
    return {i: done[i] for i in ids}


In [ ]:
%%writefile scripts/profile_corpus.py
"""Week 2: profile every document and query of a task with the frozen prompts (vLLM, Kaggle 2x T4).

Runs one vLLM process per GPU, each on every n-th id, writing to its own shard file; the parent
merges shards into the cache when they finish. Already-cached ids are skipped, so a run killed
by Kaggle's time limit resumes where it stopped. Wall time per run goes to cost.jsonl in the
cache folder (GPU minutes = seconds x gpus / 60).

    python scripts/profile_corpus.py --task writing --limit 200   # timing pilot (first 200 docs)
    python scripts/profile_corpus.py --task writing               # everything
    python scripts/profile_corpus.py --task math --setting math_soln --kinds doc
    python scripts/profile_corpus.py --task math --model Qwen/Qwen2.5-3B-Instruct-AWQ   # ablation A2
"""
import argparse
import json
import os
import subprocess
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from obliq.data import load_task
from obliq.profiles import PROFILES, VLLMBackend, load_profiles, merge_shards, profile, variant_name

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"


def ids_for(task, kind, limit):
    # sorted so the pilot's first 200 docs are also the first 200 of the full run (no rework)
    ids = sorted(task.corpus if kind == "doc" else task.queries)
    return ids[:limit] if limit else ids


def worker(args):
    """One GPU: profile ids[shard::nshards] for each kind."""
    task = load_task(args.task)
    variant = variant_name(args.setting, args.model, VLLMBackend.decoding)
    backend = VLLMBackend(args.model)
    for kind in args.kinds:
        mine = ids_for(task, kind, args.limit)[args.shard::args.nshards]
        profile(task, kind, mine, backend, variant, args.model, args.setting, shard=args.shard)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--task", required=True, choices=["math", "writing"])
    ap.add_argument("--setting", default=None, help="math_soln = Math docs profiled with solutions")
    ap.add_argument("--model", default=MODEL, help="profiler LLM (ablation A2: Qwen/Qwen2.5-3B-Instruct-AWQ)")
    ap.add_argument("--kinds", nargs="+", default=["query", "doc"], choices=["query", "doc"])
    ap.add_argument("--limit", type=int, default=0, help="only the first N ids per kind (timing pilot)")
    ap.add_argument("--gpus", type=int, default=None, help="default: all visible GPUs")
    ap.add_argument("--shard", type=int, default=None, help=argparse.SUPPRESS)
    ap.add_argument("--nshards", type=int, default=1, help=argparse.SUPPRESS)
    args = ap.parse_args()
    args.setting = args.setting or args.task
    if args.shard is not None:
        return worker(args)

    task = load_task(args.task)
    variant = variant_name(args.setting, args.model, VLLMBackend.decoding)
    vdir = PROFILES / task.name / variant
    vdir.mkdir(parents=True, exist_ok=True)  # shard logs and cost.jsonl go here before any profile exists
    for kind in args.kinds:  # fold in shards from a previous killed run before counting
        merge_shards(task.name, variant, kind)
    todo = {k: [i for i in ids_for(task, k, args.limit) if i not in load_profiles(task.name, variant, k)]
            for k in args.kinds}
    print(f"{variant}: to generate " + ", ".join(f"{k}={len(v)}" for k, v in todo.items()), flush=True)
    if not any(todo.values()):
        return

    import torch
    gpus = args.gpus or max(torch.cuda.device_count(), 1)
    t0 = time.time()
    procs = []
    for g in range(gpus):
        cmd = [sys.executable, __file__, "--task", args.task, "--setting", args.setting,
               "--model", args.model, "--kinds", *args.kinds, "--limit", str(args.limit), "--shard", str(g), "--nshards", str(gpus)]
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(g)}
        # GPU 0 shows its progress bar here; the others log to a file
        log = None if g == 0 else open(vdir / f"shard{g}.log", "w")
        procs.append((subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT if log else None), log))
    codes = [p.wait() for p, _ in procs]
    for _, log in procs:
        if log:
            log.close()
    secs = time.time() - t0

    for kind in args.kinds:
        merge_shards(task.name, variant, kind)
    left = {k: sum(i not in load_profiles(task.name, variant, k) for i in ids_for(task, k, args.limit))
            for k in args.kinds}
    n = sum(len(v) for v in todo.values()) - sum(left.values())
    rec = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "setting": args.setting, "model": args.model, "kinds": args.kinds,
           "limit": args.limit, "generated": n, "seconds": round(secs, 1), "gpus": gpus,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
           "exit_codes": codes}
    with open(vdir / "cost.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")
    # includes vLLM start-up and model load, so the pilot overstates the per-item cost a little
    print(f"\ngenerated {n} in {secs / 60:.1f} min on {gpus} GPU(s) = {secs / max(n, 1):.2f} s/item wall, "
          f"{secs * gpus / 60:.1f} GPU-min; still missing {left}", flush=True)
    if args.limit:
        total = len(task.corpus) + len(task.queries)
        print(f"extrapolated full {args.setting} ({total} items): ~{secs / max(n, 1) * total / 3600:.1f} h wall")
    if any(codes) or any(left.values()):
        sys.exit(f"incomplete: exit codes {codes}, missing {left} (re-run to resume)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile prompts/math_problem.txt
Read the math problem below. Do not solve it in full. Work out the core reasoning move a solution needs (the "aha moment") and describe it so that a mathematician would recognise the same move in a problem from a completely different field.
Name REASONING TECHNIQUES, not the topic. Do NOT mention the field or the objects involved (no words such as triangle, polynomial, matrix, prime, graph, integral, sequence). Only name a technique if a solution really needs it.
Write in words only: no formulas, symbols, LaTeX, or variable names.
Answer in exactly this format, 60-100 words in total:
Techniques: <2-4 technique names>
Proof move: <3-5 abstract steps joined by "->">
Key insight: <one abstract sentence>
Also known as: <2-3 other ways a mathematician might phrase the same move>

Problem:
{text}


In [ ]:
%%writefile prompts/math_problem_short.txt
Read the math problem below. Do not solve it in full. Work out the core reasoning move a solution needs (the "aha moment") and describe it so that a mathematician would recognise the same move in a problem from a completely different field.
Name REASONING TECHNIQUES, not the topic. Do NOT mention the field or the objects involved (no words such as triangle, polynomial, matrix, prime, graph, integral, sequence). Only name a technique if a solution really needs it.
Write in words only: no formulas, symbols, LaTeX, or variable names.
Answer in exactly this format, about 40 words in total:
Techniques: <1-2 technique names>
Proof move: <2-3 abstract steps joined by "->">
Key insight: <one abstract sentence>
Also known as: <1 other way a mathematician might phrase the same move>

Problem:
{text}


In [ ]:
%%writefile prompts/math_problem_solution.txt
Read the math problem and its solution below. Identify the core reasoning move the solution uses (the "aha moment") and describe it so that a mathematician would recognise the same move in a problem from a completely different field.
Name REASONING TECHNIQUES, not the topic. Do NOT mention the field or the objects involved (no words such as triangle, polynomial, matrix, prime, graph, integral, sequence). Only name a technique the solution really uses.
Write in words only: no formulas, symbols, LaTeX, or variable names.
Answer in exactly this format, 60-100 words in total:
Techniques: <2-4 technique names>
Proof move: <3-5 abstract steps joined by "->">
Key insight: <one abstract sentence>
Also known as: <2-3 other ways a mathematician might phrase the same move>

Problem:
{problem}

Solution:
{solution}


In [ ]:
%%writefile prompts/math_problem_topic.txt
Read the math problem below. Do not solve it in full. Work out the core reasoning move a solution needs (the "aha moment") and describe it so that a mathematician would recognise the same move in a problem from a completely different field.
Name the reasoning techniques, and also name the field and the objects involved (for example triangle, polynomial, matrix, prime, graph, integral, sequence). Only name a technique if a solution really needs it.
Write in words only: no formulas, symbols, LaTeX, or variable names.
Answer in exactly this format, 60-100 words in total:
Techniques: <2-4 technique names>
Proof move: <3-5 abstract steps joined by "->">
Key insight: <one abstract sentence>
Also known as: <2-3 other ways a mathematician might phrase the same move>

Problem:
{text}


In [ ]:
%%writefile prompts/rewrite_math.txt
You are helping build an information retrieval benchmark focused on math problems that share a REASONING TECHNIQUE.
The task: given a query problem, retrieve other problems whose solutions need the same core reasoning pattern, the same "aha moment" or key insight, even if they come from completely different areas of mathematics. The relevant problems will likely share little vocabulary with the query.
Rewrite the query into a form that will perform better with an embedding model:
- Capture the core reasoning pattern a solution would need
- Phrase it so that it would match problems from other areas that need the same insight
- Remain a query (not a solution)
- Stay concise (1-3 sentences)
- Avoid verbose academic language

ORIGINAL QUERY: {text}


In [ ]:
%%writefile prompts/rewrite_writing.txt
You are an expert at author attribution.
Given a text snippet, rewrite the query to better capture the distinctive stylistic features that would help find other texts by the same author. Focus on vocabulary patterns, sentence structure, tone, and other authorial fingerprints.
Return: {{"rewritten_query": "...", "rationale": "..."}}

ORIGINAL TEXT: {text}
Rewrite to capture the author's distinctive style.


In [ ]:
%%writefile prompts/writing_style.txt
You are a stylometry expert. Describe what makes the AUTHOR of the passage below recognisable, so that the same author could be identified from a passage on a completely different topic.
Focus on DISTINCTIVE habits. Skip traits that most blog posts share (such as "short sentences", "technical vocabulary", "informative tone", "uses commas") unless they are unusually strong here.
Do NOT mention the subject matter, names, places, products, or facts from the passage.
Answer in exactly this format, 60-100 words in total:
Voice: <person (I/we/you), stance toward the reader, formality>
Sentences: <typical length, rhythm, how sentences open and end>
Punctuation: <distinctive punctuation habits>
Tone and humour: <a few words>
Signature phrases: <2-4 short non-topical phrases or verbal tics quoted from the passage, such as hedges, intensifiers or transitions>
Other habits: <spelling, capitalisation, contractions, slang, formatting, paragraph structure>

Passage:
{text}


## A1 topic words allowed

In [ ]:
!python scripts/profile_corpus.py --task math --setting math_topic
!zip -qr /kaggle/working/ablation_out.zip profiles && ls -la /kaggle/working/ablation_out.zip

## A3 ~40-word profiles

In [ ]:
!python scripts/profile_corpus.py --task math --setting math_short
!zip -qr /kaggle/working/ablation_out.zip profiles && ls -la /kaggle/working/ablation_out.zip

## A2 3B profiler (same prompt as the main system)

In [ ]:
!python scripts/profile_corpus.py --task math --model Qwen/Qwen2.5-3B-Instruct-AWQ
!zip -qr /kaggle/working/ablation_out.zip profiles && ls -la /kaggle/working/ablation_out.zip

In [ ]:
# eyeball: the first two doc profiles of each variant
import glob, json
for f in sorted(glob.glob('profiles/math/*/docs.jsonl')):
    print('==', f)
    for line in open(f).readlines()[:2]:
        print(json.loads(line)['profile'], '\n')
!cat profiles/*/*/cost.jsonl